# 4 · plot — decontamination contrast

Draws [`4_make_contamination_contrast_data.ipynb`](4_make_contamination_contrast_data.ipynb):
the paired difference between the decontaminated checkpoint and the contaminated one, overall and
by how close each protein is to the un-decontaminated training set.

A leakage story predicts a gap that grows with identity and vanishes without a homolog. Read the
`no_homolog` bar next to its n and next to the two absolute means printed below — those proteins
are much harder for both models, so a small gap there is not by itself evidence.

In [ ]:
# Run from anywhere: figlib lives next to this notebook.
import sys
from pathlib import Path

HERE = Path.cwd() if (Path.cwd() / "figlib.py").exists() else Path("notebooks/figures")
sys.path.insert(0, str(HERE.resolve()))
import figlib

In [ ]:
DATASET = "4_contamination_contrast"
DPI = 300
STRATUM_ORDER = ["no_homolog", "id_20_30", "id_30_50", "id_50_70", "id_70_100"]
STRATUM_LABELS = {"no_homolog": "no homolog", "id_20_30": "20–30%", "id_30_50": "30–50%",
                  "id_50_70": "50–70%", "id_70_100": "70–100%"}
metadata = figlib.describe(DATASET)

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

figlib.figure_style(DPI)
directory = figlib.require(DATASET, "summary.csv", "per_protein.csv")
summary = pd.read_csv(directory / "summary.csv")
per_protein = pd.read_csv(directory / "per_protein.csv")
focus = metadata["contrast"]["focus"]
baseline = metadata["contrast"]["baseline"]

strata = summary[summary.group.isin(STRATUM_ORDER)].set_index("group").reindex(
    [s for s in STRATUM_ORDER if s in set(summary.group)])
overall = summary[summary.group == "all"].iloc[0]
print(f"overall {overall.delta:+.3f} [{overall.ci_low:+.3f}, {overall.ci_high:+.3f}]  "
      f"n={int(overall.n)}  spearman "
      f"{metadata['result']['spearman_delta_vs_identity']:+.3f}")

In [ ]:
FOCUS_LABEL = focus.replace(" (decontaminated)", ", decontaminated").lstrip("#").strip()
BASELINE_LABEL = baseline.replace(" (contaminated)", ", contaminated").lstrip("#").strip()

figure, axes = plt.subplots(1, 2, figsize=(7.6, 3.2), width_ratios=[1.25, 1],
                            layout="constrained")

# Left: the paired difference per stratum, with the overall value as a reference line.
positions = range(len(strata))
axes[0].bar(positions, strata.delta, width=0.66, color="#C44E52",
            yerr=[strata.delta - strata.ci_low, strata.ci_high - strata.delta],
            error_kw=dict(ecolor="0.25", lw=0.9, capsize=2.5))
axes[0].axhline(0, color="0.2", lw=0.9)
axes[0].axhline(overall.delta, color="0.45", lw=0.9, ls="--")
axes[0].text(len(strata) - 0.5, overall.delta, f"  all proteins {overall.delta:+.3f}",
             va="center", ha="left", fontsize=7.5, color="0.35")
axes[0].set_xticks(list(positions))
axes[0].set_xticklabels([f"{STRATUM_LABELS.get(s, s)}\nn={int(n)}"
                         for s, n in zip(strata.index, strata.n)])
axes[0].set(xlabel="identity to the un-decontaminated training set",
            ylabel=f"{FOCUS_LABEL} −\n{BASELINE_LABEL}   (R-precision)")
axes[0].grid(axis="y", alpha=0.25, lw=0.6)
axes[0].set_axisbelow(True)

# Right: the two absolute means per stratum, which is what says whether a small gap is a floor.
width = 0.38
# Labels come from the dataset, so changing FOCUS/BASELINE cannot mislabel.
axes[1].bar([p - width / 2 for p in positions], strata.focus, width, label=FOCUS_LABEL,
            color="#C44E52")
axes[1].bar([p + width / 2 for p in positions], strata.baseline, width, label=BASELINE_LABEL,
            color="#7A8DA6")
axes[1].set_xticks(list(positions))
axes[1].set_xticklabels([STRATUM_LABELS.get(s, s) for s in strata.index], rotation=20, ha="right")
axes[1].set(xlabel="identity to the un-decontaminated training set", ylabel="R-precision",
            ylim=(0, 1.0))
axes[1].legend(frameon=False, loc="upper left")
axes[1].grid(axis="y", alpha=0.25, lw=0.6)
axes[1].set_axisbelow(True)

figlib.save_figure(figure, "contamination_contrast", DPI)
plt.show()

In [ ]:
# The per-protein scatter behind those bars: where the two checkpoints actually disagree.
figure, axis = plt.subplots(figsize=(4.0, 4.0))
axis.scatter(per_protein[baseline], per_protein[focus], s=22, alpha=0.75, color="#4C72B0",
             edgecolor="none")
axis.plot([0, 1], [0, 1], color="0.4", lw=1, ls="--")
axis.set(xlabel=f"{BASELINE_LABEL} · R-precision", ylabel=f"{FOCUS_LABEL} · R-precision",
         xlim=(0, 1.02), ylim=(0, 1.02))
axis.grid(alpha=0.25, lw=0.6)
axis.set_axisbelow(True)
figlib.save_figure(figure, "contamination_contrast_scatter", DPI)
plt.show()
print(f"decontaminated ahead on {100 * (per_protein.delta > 0).mean():.0f}% of "
      f"{len(per_protein)} proteins")